# Wind Sensitivity Analysis — 155 mm Shell

Runs **200 Monte-Carlo simulations** with random crosswind to quantify impact-point dispersion and compute the **Circular Error Probable (CEP)**.

### Method

Each run uses the full 6-DOF model (Mach-dependent $C_D$, aero moments, Magnus force) with a crosswind component drawn from:

$$w_y \sim \mathcal{N}(0, \sigma^2), \quad \sigma = 5 \text{ m/s}$$

### CEP Definition

The **Circular Error Probable** is the radius of the smallest circle, centred on the mean impact point, that contains **50%** of all impact points. It is the standard measure of weapon system accuracy.

$$\text{CEP} \approx \text{median}(r_i)$$

where $r_i = \sqrt{(x_i - \bar{x})^2 + (y_i - \bar{y})^2}$.

In [ ]:
import sys, math
import numpy as np
import matplotlib.pyplot as plt
sys.path.insert(0, '..')

from src.atmosphere import ISAAtmosphere
from src.aerodynamics import AeroModel
from src.integrator import TrajectoryIntegrator
from src.projectile_config import SHELL_155MM

In [ ]:
N_RUNS = 200
WIND_STDDEV_MS = 5.0
MUZZLE_VELOCITY_MS = 827.0
ELEVATION_DEG = 45.0
SPIN_RATE_RAD_S = 1800.0
DT_S = 0.05
SIM_TIME_S = 300.0

atmosphere = ISAAtmosphere()
config = SHELL_155MM
aero = AeroModel(
    Cd=config['Cd'], Cl=config['Cl'], Cm=config['Cm'],
    reference_area_m2=config['reference_area_m2'],
    reference_diameter_m=config['diameter_m'],
    Cmq=config['Cmq'], Clp=config['Clp'], Cnpa=config['Cnpa'],
    mach_table=config['mach_table'],
    cd_table=config['cd_table'],
    reference_length_m=config['length_m'],
)
integrator = TrajectoryIntegrator()

elev_rad = math.radians(ELEVATION_DEG)
half = elev_rad / 2.0
base_state = np.array([
    0.0, 0.0, 0.0,
    MUZZLE_VELOCITY_MS * math.cos(elev_rad), 0.0,
    MUZZLE_VELOCITY_MS * math.sin(elev_rad),
    math.cos(half), 0.0, -math.sin(half), 0.0,
    SPIN_RATE_RAD_S, 0.0, 0.0,
])

print(f'Running {N_RUNS} Monte-Carlo simulations with σ_wind = {WIND_STDDEV_MS} m/s …')

In [ ]:
np.random.seed(42)

impact_x = np.zeros(N_RUNS)
impact_y = np.zeros(N_RUNS)

for i in range(N_RUNS):
    crosswind_ms = np.random.normal(0.0, WIND_STDDEV_MS)
    wind = np.array([0.0, crosswind_ms, 0.0])

    result = integrator.integrate(
        initial_state=base_state.copy(),
        t_span=(0.0, SIM_TIME_S),
        dt=DT_S,
        mass_kg=config['mass_kg'],
        inertia_tensor=config['inertia_tensor'],
        aero_model=aero,
        atmosphere=atmosphere,
        wind_vector_ms=wind,
    )

    impact_x[i] = result.state[-1, 0]
    impact_y[i] = result.state[-1, 1]

    if (i + 1) % 50 == 0:
        print(f'  {i + 1}/{N_RUNS} complete')

print('\n✅ All runs complete.')

In [ ]:
mean_x = np.mean(impact_x)
mean_y = np.mean(impact_y)
radii = np.sqrt((impact_x - mean_x)**2 + (impact_y - mean_y)**2)
CEP_m = np.median(radii)

print(f'Mean impact point  : ({mean_x:,.1f} m, {mean_y:,.1f} m)')
print(f'Mean range         : {mean_x/1000:,.2f} km')
print(f'CEP (50% radius)   : {CEP_m:,.1f} m')
print(f'Max cross-range    : {np.max(np.abs(impact_y)):,.1f} m')
print(f'σ cross-range      : {np.std(impact_y):,.1f} m')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

ax.scatter(impact_x / 1000, impact_y, s=20, alpha=0.6, c='steelblue',
           edgecolors='none', label='Impact points')
ax.scatter(mean_x / 1000, mean_y, s=120, c='red', marker='x',
           linewidths=2.5, label='Mean impact')

# CEP circle
theta = np.linspace(0, 2 * np.pi, 200)
ax.plot(mean_x / 1000 + CEP_m / 1000 * np.cos(theta),
        mean_y + CEP_m * np.sin(theta),
        'r--', linewidth=1.5, label=f'CEP = {CEP_m:,.0f} m')

ax.set_xlabel('Down-range [km]')
ax.set_ylabel('Cross-range [m]')
ax.set_title(f'155 mm Shell — Impact Dispersion ({N_RUNS} runs, σ_wind = {WIND_STDDEV_MS} m/s)')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()